# Model Experiments

## Objective

The objective of this notebook is to build and compare different content-based recommendation approaches for the movie recommendation system. The goal is to identify the vectorization technique that produces the most relevant movie recommendations.

## Input

The notebook uses the cleaned and preprocessed dataset generated from `01_eda_cleaning.ipynb`.

## Workflow

1. Load cleaned dataset
2. Perform feature engineering
3. Create the `tags` feature
4. Apply text preprocessing
5. Experiment with different vectorization techniques
6. Compute similarity scores
7. Compare recommendation quality
8. Save the best model artifacts

In [1]:
import pandas as pd
import numpy as np

In [2]:
new_df=pd.read_csv("../data/processed/movies_processed.csv")

In [3]:
new_df.head()

,movie_id,title,tags
0,19995,Avatar,"in the 22nd century, a paraplegic marine is di..."
1,285,Pirates of the Caribbean: At World's End,"captain barbossa, long believed to be dead, ha..."
2,206647,Spectre,a cryptic message from bond’s past sends him o...
3,49026,The Dark Knight Rises,following the death of district attorney harve...
4,49529,John Carter,"john carter is a war-weary, former military ca..."


In [4]:
new_df["tags"].values[0]

'in the 22nd century, a paraplegic marine is dispatched to the moon pandora on a unique mission, but becomes torn between following orders and protecting an alien civilization. action adventure fantasy sciencefiction cultureclash future spacewar spacecolony society spacetravel futuristic romance space alien tribe alienplanet cgi marine soldier battle loveaffair antiwar powerrelations mindandsoul 3d samworthington zoesaldana sigourneyweaver jamescameron'

## Text Preprocessing

Before applying vectorization, the textual data needs to be normalized. Different grammatical forms of the same word (e.g., *accept*, *accepted*, *accepting*) represent the same underlying concept but would be treated as separate features by the vectorizer.

To reduce feature redundancy and improve the quality of the vocabulary, stemming is applied to convert words to their root form.

In [5]:
import re
from nltk.stem.porter import PorterStemmer

ps = PorterStemmer()

def stem(text):
    words = re.findall(r"\b\w+\b", text.lower())

    stemmed_words = []

    for word in words:
        stemmed_words.append(ps.stem(word))

    return " ".join(stemmed_words)

In [6]:
new_df["tags"] = new_df["tags"].apply(stem)

In [7]:
new_df["tags"]

0       in the 22nd centuri a parapleg marin is dispat...
1       captain barbossa long believ to be dead ha com...
2       a cryptic messag from bond s past send him on ...
3       follow the death of district attorney harvey d...
4       john carter is a war weari former militari cap...
                              ...                        
4795    el mariachi just want to play hi guitar and ca...
4796    a newlyw coupl s honeymoon is upend by the arr...
4797    sign seal deliv introduc a dedic quartet of ci...
4798    when ambiti new york attorney sam is sent to s...
4799    ever sinc the second grade when he first saw h...
Name: tags, Length: 4800, dtype: object

### Observation

After stemming, multiple variations of the same word are reduced to a common root. This decreases the vocabulary size, removes redundant features, and helps the vectorizer focus on semantic information rather than grammatical variations.

## Bag of Words Vectorization

Machine learning algorithms cannot process raw text directly. Therefore, the preprocessed `tags` column is converted into numerical feature vectors using the Bag of Words technique.

`CountVectorizer` builds a vocabulary of the most frequent words and represents each movie as a vector containing the frequency of those words.

In [8]:
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer(max_features=5000,stop_words="english")
vectors = cv.fit_transform(new_df["tags"]).toarray()

In [9]:
vectors

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], shape=(4800, 5000))

In [10]:
feature_names = cv.get_feature_names_out()
feature_names[:100]

array(['000', '007', '10', '100', '11', '12', '13', '14', '15', '16',
       '17', '17th', '18', '18th', '18thcenturi', '19', '1910', '1920',
       '1930', '1940', '1944', '1950', '1960', '1970', '1971', '1974',
       '1976', '1980', '1985', '1990', '1999', '19th', '19thcenturi',
       '20', '200', '2000', '2003', '2009', '20th', '21st', '23', '24',
       '25', '30', '300', '3d', '40', '50', '500', '60', '70', '80',
       'aaron', 'aaroneckhart', 'abandon', 'abbi', 'abduct',
       'abigailbreslin', 'abil', 'abl', 'aboard', 'aborigin', 'abov',
       'absenc', 'abus', 'academ', 'academi', 'accept', 'access', 'accid',
       'accident', 'acclaim', 'accompani', 'accomplish', 'account',
       'accus', 'ace', 'achiev', 'acquaint', 'act', 'action',
       'actionhero', 'activ', 'activist', 'actor', 'actress', 'actual',
       'ad', 'adam', 'adamsandl', 'adamshankman', 'adapt', 'add',
       'addict', 'addit', 'adjust', 'admir', 'admit', 'adolesc', 'adopt'],
      dtype=object)

In [11]:
print("Feature Matrix Shape:", vectors.shape)
print("Vocabulary Size:", len(cv.get_feature_names_out()))

Feature Matrix Shape: (4800, 5000)
Vocabulary Size: 5000


## Results of Bag of Words Vectorization

The `tags` column has been transformed into numerical feature vectors using CountVectorizer.

- Total movies: 4,806
- Vocabulary size: 5,000 most frequent words
- Stop words removed
- Each movie is represented as a sparse vector of length 5,000.

This numerical representation enables mathematical comparison between movies based on their textual features.

## Similarity Computation using Cosine Similarity

In [12]:
from sklearn.metrics.pairwise import cosine_similarity
similarity = cosine_similarity(vectors)

In [13]:
similarity.shape

(4800, 4800)

## Cosine Similarity Matrix

Cosine Similarity measures how similar two movie vectors are based on the angle between them rather than their absolute magnitude.

The resulting similarity matrix has shape (4806, 4806).

- Rows represent source movies.
- Columns represent comparison movies.
- Each value ranges from 0 to 1.
    - 1 → Identical movie representation.
    - 0 → Completely unrelated.

This matrix forms the core of the recommendation engine.

# Building the Movie Recommendation Function

With the similarity matrix computed, the next step is to build a recommendation function.

The function takes a movie title as input, finds its position in the dataset, retrieves similarity scores with all other movies, sorts them in descending order, and returns the five most similar movies.

Since every movie has a similarity score with every other movie, the movie itself (similarity = 1.0) is excluded from the recommendations.

In [14]:
sorted(list(enumerate(similarity[0])),reverse=True ,key=lambda x:x[1])[1:6]

[(2403, np.float64(0.2597056905574371)),
 (3723, np.float64(0.2506402059138015)),
 (539, np.float64(0.24715576637149034)),
 (1213, np.float64(0.2464320290450721)),
 (507, np.float64(0.24403555043462524))]

In [15]:
def recommend(movie):
    movie_index=new_df[new_df["title"] == movie].index[0]
    distances = similarity[movie_index]
    movie_list=sorted(list(enumerate(distances)),reverse=True ,key=lambda x:x[1])[1:6]
    
    for i in movie_list:
        print(new_df.iloc[i[0]].title)

In [25]:
new_df[new_df["title"]=="The Impossible"].values[0]

array([80278, 'The Impossible',
       'in decemb 2004 close knit famili maria henri and their three son begin their winter vacat in thailand but the day after christma the idyl holiday turn into an incomprehens nightmar when a terrifi roar rise from the depth of the sea follow by a wall of black water that devour everyth in it path though maria and her famili face their darkest hour unexpect display of kind and courag amelior their terror thriller drama thailand tsunami familyvac tidalwav catastroph sweptaway separationfromfamili boxingday 21stcenturi naomiwatt ewanmcgregor tomholland juanantoniobayona'],
      dtype=object)

In [24]:
recommend("The Impossible")


Love the Coopers
Insidious: Chapter 3
Funny Games
Escobar: Paradise Lost
This Christmas


# Saving the Processed Data

The recommendation system has been successfully built and tested.

To avoid repeating preprocessing and similarity computation every time the application starts, the processed movie dataset and cosine similarity matrix are serialized using Python's `pickle` module.

These files will later be loaded directly into the Streamlit application for fast recommendations.

In [28]:
from pathlib import Path
import pickle

ARTIFACTS_DIR = Path("../artifacts")
ARTIFACTS_DIR.mkdir(exist_ok=True)

with open(ARTIFACTS_DIR / "movies.pkl", "wb") as f:
    pickle.dump(new_df, f)

with open(ARTIFACTS_DIR / "similarity.pkl", "wb") as f:
    pickle.dump(similarity, f)